组合计数
1. 基础计数： 处理元素与容器的区分度（球盒模型）- Twelvefold Way
2. 约束计数： 处理容斥原理、抽屉原理（约束条件）。
3. 等价计数： 处理群论下的对称性（Burnside & Pólya）。

### Twelvefold Way -> 二十四路计数模型

**Bell数**的整数数列：B0=1,B1=1,B2=2,B3=5,B4=15,B5=52,B6=203.... , Bn是基数为n的集合的划分方法数目。
- 集合S的一个划分partition是定义为S的两两不相交的非空子集的族，他们的并是S。
- B3=5，集合S={1,2,3}的划分就是: 
{{1},{2},{3}}
{{1,2},{3}}
{{1,3},{2}}
{{2,3},{1}}
{{1,2,3}}.

**第二类Stirling数**{S(n, k)}：将 n 个不同元素分到 k 个相同非空集合中的方法数。 
- S(3, 2) = 3，将 {1, 2, 3} 分为 2 个非空集，有 {{1,2},{3}}, {{1,3},{2}}, {{1},{2,3}} 三种方案。
- 递推公式为：
S(n,n+1)=0
S(n,0)=0
S(0,0)=1
S(n,k)=kS(n-1,k)+S(n-1,k-1)
- Bell数 =$\sum$ 每行S(n,k)

| n\k| 0|1|2|3|4|5
|---|---|---|---|---|-|-|
|0|1|0|0|0|0|0|
|1|0|1|0|0|0|0|
|2|0|1|1|0|0|0|
|3|0|1|3|1|0|0|
|4|0|1|7|6|1|0|
|5|0|1|15|25|10|1|

**Partition分拆函数** p(n)，从 n=0开始的序列是 1, 1, 2, 3, 5, 7, 11, 15, 22, 30, 42, 56, 77, ..
定义：数字n被拆分成多少种不同组合（不考虑顺序）。
- n=3共有 3 种分拆：3, 2+1, 1+1+1，p(3)=3
- n=4共有 5 种分拆：4, 3+1, 2+2, 2+1+1, 1+1+1+1，p(4)=5



| Ball $N$ | Box $K$ | $f$ unrestricted | $f$ one-to-one injective| $f$ onto Surjective|
|---|---|---|---|---|
| labeled | labeled | $k^n$ | $(k-n+1)_n$ | $k!\,S(n,k)$ |
| unlabeled | labeled | $\binom{k+n-1}{n}$ | $\binom{k}{n}$ | $\binom{n-1}{\,n-k\,}$ |
| labeled | unlabeled | $S(n,1)+S(n,2)+\cdots+S(n,k)$ | $\begin{cases}1 & n\le k\\0 & n>k\end{cases}$ | $S(n,k)$ |
| unlabeled | unlabeled | $p_k(n)$ | $\begin{cases}1 & n\le k\\0 & n>k\end{cases}$ | $p_k(n)-p_{k-1}(n)$ |

二十四路计数的三个逻辑维度
- 元素（球）是否可区分？ (Distinguishable / Indistinguishable)
- 容器（盒）是否可区分？ (Distinguishable / Indistinguishable)
- 容器内的元素是否有序？ (Ordered / Unordered) —— 这是 24 路相比 12 路增加的关键维度

ps: Twelvefold Way是所有计数的核心，剩下的“有序”情况(即24路)可以通过“先分组、再排列”推导出来

Structural summary
  - Stirling numbers = unlabeled blocks
  - Bell numbers = all partitions
  - Partitions = size-only structure


球是否相同|盒是否相同|盒内是否有序|无限制 (L)|单射 (I)|满射 (S)
|-|-|-|-|-|-|
不同|不同|无序|m^n|"P(m,n)"|"m!S2​(n,m)"
相同|不同|无序|"C(n+m−1,m−1)"|"C(m,n)"|"C(n−1,m−1)"
不同|相同|无序|"∑S2​(n,k)"|[n≤m]|"S2​(n,m)"
相同|相同|无序|"p(n,m)"|[n≤m]|"p(n−m,m)"
不同|不同|有序|"P(n+m−1,n)"|"P(m,n)"|"n!C(n−1,m−1)"
不同|相同|有序|(分配并除以 m!)|[n≤m]|(同左)

In [8]:
# Twelvefold Way in SageMath (n balls --> k boxes)
def TF_distinctBalls_labeledBoxes(n, k): #     Balls distinct/labeled, boxes labeled.
    any = k^n                                                 # k^n , unrestricted
    inj = 0 if n > k else factorial(k) // factorial(k - n)    # (k)_n, falling_factorial(k,n)
    sur = factorial(k) * stirling_number2(n, k)               # k! S(n,k)
    return any, inj, sur

def TF_identicalBalls_labeledBoxes(n, k): # Occupancy is vector
    any = binomial(n + k - 1, k - 1)              # xi>=0, sum=n -> stars and bars C(n+k-1,k-1)
    inj = 0 if n > k else binomial(k, n)          # xi in {0,1}, sum=n -> choose n boxes
    sur = 0 if n < k else binomial(n - 1, k - 1)  # xi>=1, sum=n -> compositions count C(n-1,k-1)
    return any, inj, sur

def TF_distinctBalls_unlabeledBoxes(n, k): #   Balls distinct, boxes unlabeled/identical
    any = sum(stirling_number2(n, i) for i in range(0, k+1)) # set partitions into <= k blocks -> sum_{i=0..k} S(n,i)
    inj = 1 if n <= k else 0     # all balls separated (possible iff n<=k)
    sur = stirling_number2(n, k) # set partitions into exactly k blocks -> S(n,k)
    return any, inj, sur

def TF_identicalBalls_unlabeledBoxes(n, k): # Occupancy is a partition (multiset of block sizes)
    any = Partitions(n, max_length=k).cardinality()  # partitions of n into <= k parts -> p_{<=k}(n)
    inj = 1 if n <= k else 0                         # pattern is 1+...+1 (n ones) (possible iff n<=k)
    sur = 0 if n < k else Partitions(n, length=k).cardinality() # partitions of n into k positive parts -> p_k(n)
    return any, inj, sur

def twelvefold_summary(n, k): # print a neat 12-way summary for given n,k
    db_lb = TF_distinctBalls_labeledBoxes(n, k)
    db_ub = TF_distinctBalls_unlabeledBoxes(n, k)
    ib_lb = TF_identicalBalls_labeledBoxes(n, k)
    ib_ub = TF_identicalBalls_unlabeledBoxes(n, k)
    print("")
    print("Twelvefold way (", n, "balls, ",k, "boxes) with capacity rule:")
    print("1) Balls labeled,   Boxes labeled:   Any:", db_lb[0], "  Inj:", db_lb[1], "  Sur:", db_lb[2])
    print("2) Balls labeled,   Boxes unlabeled: Any:", db_ub[0], "  Inj:", db_ub[1], "  Sur:", db_ub[2])
    print("3) Balls unlabeled, Boxes labeled:   Any:", ib_lb[0], "  Inj:", ib_lb[1], "  Sur:", ib_lb[2])
    print("4) Balls unlabeled, Boxes unlabeled: Any:", ib_ub[0], "  Inj:", ib_ub[1], "  Sur:", ib_ub[2])

# Optional: small brute checks (useful for sanity for small n,k)
def brute_distinctBalls_labeledBoxes_any(n, k):
    return k^n      # Count all functions [n]->[k]
def brute_distinctBalls_labeledBoxes_inj(n, k):
    if n > k: return 0      # injections = permutations of k choose n = (k)_n
    return factorial(k) // factorial(k - n)
def brute_distinctBalls_labeledBoxes_sur(n, k):
    return factorial(k) * stirling_number2(n, k) # surj count via Stirling is already exact; brute is expensive

# 3 examples 
print("           Any=unrestricted;   Injective=at most 1 per box;  Surjective=no empty box")
twelvefold_summary(5, 5) # it is bijective 
twelvefold_summary(5, 3)
twelvefold_summary(5, 6)

           Any=unrestricted;   Injective=at most 1 per box;  Surjective=no empty box

Twelvefold way ( 5 balls,  5 boxes) with capacity rule:
1) Balls labeled,   Boxes labeled:   Any: 3125   Inj: 120   Sur: 120
2) Balls labeled,   Boxes unlabeled: Any: 52   Inj: 1   Sur: 1
3) Balls unlabeled, Boxes labeled:   Any: 126   Inj: 1   Sur: 1
4) Balls unlabeled, Boxes unlabeled: Any: 7   Inj: 1   Sur: 1

Twelvefold way ( 5 balls,  3 boxes) with capacity rule:
1) Balls labeled,   Boxes labeled:   Any: 243   Inj: 0   Sur: 150
2) Balls labeled,   Boxes unlabeled: Any: 41   Inj: 0   Sur: 25
3) Balls unlabeled, Boxes labeled:   Any: 21   Inj: 0   Sur: 6
4) Balls unlabeled, Boxes unlabeled: Any: 5   Inj: 0   Sur: 2

Twelvefold way ( 5 balls,  6 boxes) with capacity rule:
1) Balls labeled,   Boxes labeled:   Any: 7776   Inj: 720   Sur: 0
2) Balls labeled,   Boxes unlabeled: Any: 52   Inj: 1   Sur: 0
3) Balls unlabeled, Boxes labeled:   Any: 252   Inj: 6   Sur: 0
4) Balls unlabeled, Boxes unlabeled: 

#### Stirling arise from both permutation & set partitions

```mermaid
flowchart LR
    A["Finite set {1,2,3,4,5}"] --> B[Partition into k blocks]
    A --> C[Permutation]

    %% Set partitions
    B --> B1["Block 1: {1,3,5}"]
    B --> B2["Block 2: {2,4}"]
    B1 --> D["S(n,k)"]
    B2 --> D

    %% Permutation cycles
    C --> C1["Cycle: (1 3 5)"]
    C --> C2["Cycle: (2 4)"]
    C1 --> E["c(n,k)"]
    C2 --> E

    %% Conceptual bridge
    D --> F[Unlabeled components]
    E --> F

    F --> G[Same decomposition into k components]


## Me typed


### Poker 

In [ ]:
############ Poker Simulator ############
Hua=Set(["Hearts","Diamonds","Spades","Clubs"])
Num=Set([2,3,4,5,6,7,8,9,10,"J","Q","K","A"])
Card=cartesian_product([Hua, Num]) # 4*13=52
print(Card)

s(Hua.cardinality())
s(Num.cardinality())
s(Card.cardinality())

s(Card.random_element())                      # pick up a card randomly
Set([Card.random_element(),Card.random_element()]) # two cards randomly

In [ ]:
Hands=Subsets(Card, 5)  # Five-cards game
s(Hands.random_element())
Hands.cardinality()==binomial(52,5)

Subsets(Num,5).cardinality()==binomial(13,5)    # 1287
Flushes=cartesian_product([Subsets(Num,5),Hua]) # 1287*4=5148 
s(Flushes.random_element())
Flushes.cardinality()/Hands.cardinality() # probabilty of the same Hua

### Subsets

In [ ]:
s(binomial(4,2))
A=Subsets([1,2,3,4],2); s(A.cardinality())
s(A.list())
s(A.unrank(1)) # display item (1)
s(A[1]==A.unrank(1))

In [ ]:
A=Set([1,2,3,4]); Nest=Subsets(Subsets(A)) # nested
s(Nest.cardinality())
#s(Nest.list())

### Integer partition & composition & permutation(updating)

In [48]:
P5=Partitions(5);s(P5)
s(P5.cardinality())
P5.list() # compared to Compositions(), order not exist

<IPython.core.display.Math object>

<IPython.core.display.Math object>

[[5], [4, 1], [3, 2], [3, 1, 1], [2, 2, 1], [2, 1, 1, 1], [1, 1, 1, 1, 1]]

In [ ]:
Partitions(999).cardinality()

In [ ]:
P7=Partitions(7)
p=P7.unrank(5)
s(p.ferrers_diagram())
p

In [ ]:
Partition([4,2,1]) # no 's'
WeightedIntegerVectors(8,[2,3,5]).list()

In [50]:
C5=Compositions(5); s(C5.cardinality()); s(C5.list()) # compared to Partitions(), order matters
s([Compositions(n).cardinality() for n in range(10)])
var('x')
s(sum(x^len(c) for c in C5 ))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [53]:
C=IntegerRange(3,21,2); s(C); s(C.cardinality()); s(C.list())

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [55]:
C=Permutations(4); s(C.cardinality()); s(C.list())

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [57]:
C=SetPartitions(["a","b","c"]); s(C.cardinality()); s(C.list())

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### GF 4 enumeration tree

In [ ]:
############ Start from an Integer Seq, to find out GF ############
oeis([1,1,2,5,14])

In [ ]:
# to verify A000108
var('C, z')
# method 1
sys = [C == z + C*C]
s(sys)
sol = solve(sys, C, solution_dict=True) ; #s(sol)
s0=sol[0][C]; s1=sol[1][C]
#s(s0.series(z,6), s1.series(z,6))  # Tylor series, [1][C] is wrong
#C=s0; s(C.series(z,11)); s(C.series(z,101).coefficient(z,100))

# method 2
L.<z>=LazyPowerSeriesRing(QQ)
C=L.undefined(valuation=1)
C.define(z + C*C)
s([C.coefficient(i) for i in range(11)])

z=var('z') # back to closure-type C(z)
C=s0; s(C)
s(derivative(C,z,1))

def d(n): return derivative(s0,n).subs(z=0)
[d(n+1)/d(n) for n in range(1,17)]  # d(n+1)/d(n) = 4n-2 , so C(n+1)/C(n) = (4n-2)/(n+1)

$$c_n = Catalan(n-1) = \frac{1}{n} \binom{2(n-1)}{n-1}$$

In [ ]:
n=var('n')
c=1/n*binomial(2*(n-1),n-1)
s([c.subs(n=k)         for k in range(1,11)])
s([catalan_number(k-1) for k in range(1,11)])